In [0]:
import pandas as pd

In [0]:
renewal_calls_df = pd.read_csv("../../data/raw/renewal_calls.csv")
renewal_calls_df.head()


## Drop **Duplicates**

In [0]:
renewal_calls_df = renewal_calls_df.loc[:, ~renewal_calls_df.columns.duplicated()]
renewal_calls_df.drop_duplicates(inplace=True)

## Fixing Datatypes and standardising

In [0]:
renewal_calls_df.info()

### Fixing Date Columns

In [0]:
renewal_calls_df['Call_Date'] = pd.to_datetime(renewal_calls_df['Call_Date'], dayfirst=True, errors='coerce')

   
### Check Unique Values per Column

In [0]:
# Check unique value counts and sample unique values for all columns
for col in renewal_calls_df.columns:
    n_unique = renewal_calls_df[col].nunique()
    sample_vals = renewal_calls_df[col].dropna().unique()[:10].tolist()
    print(f"{col:55s} | dtype: {str(renewal_calls_df[col].dtype):8s} | nunique: {n_unique:6d} | samples: {sample_vals}")

   
### Enforce Column Types & Convert to Categorical

   
## Fixing Datatypes and Standardising

In [0]:
# ── Fix numeric types ──
renewal_calls_df['Call_ID'] = renewal_calls_df['Call_ID'].astype('int64')
renewal_calls_df['Analysed_Call'] = renewal_calls_df['Analysed_Call'].astype('Int8')

# ── Standardise Call_Direction ──
renewal_calls_df['Call_Direction'] = renewal_calls_df['Call_Direction'].replace({
    "Outbound" : "OUT_BOUND",
    "Inbound" : "IN_BOUND"
})

# ── Drop all-null column ──
renewal_calls_df.drop(columns=['Unnamed: 20'], inplace=True)

print("Numeric types fixed, Call_Direction standardised, Unnamed: 20 dropped ✓")

In [0]:
# ── Group 1: Replace 'Yes/No' → None, then fill NaN → 'Unknown' ──
yes_no_cleanup_cols = [
    'Discussion_on_Price_Increase',
    'Membership_Renewal_Decision',
    'Serious_Complaint',
    'Other_Complaint',
]

for col in yes_no_cleanup_cols:
    renewal_calls_df[col] = (
        renewal_calls_df[col]
        .replace({'Yes/No': None})
        .fillna('Unknown')
    )


In [0]:
# ── Group 2: Replace 'Not Applicable' → 'Unknown', fill NaN → 'Unknown' ──
not_applicable_cols = [
    'Call_Reschedule_Request',
    'Agent_Flagged_Membership_Status_Alert',
    'Agent_Renewal_Initiation',
]

for col in not_applicable_cols:
    renewal_calls_df[col] = renewal_calls_df[col].replace({
        'Yes': 'Yes',
        'No': 'No',
        'Not Applicable': 'Unknown'
    }).fillna('Unknown')


In [0]:
# ── Group 3: Lowercase + strip, then keyword match → Yes / No / Unknown ──
def clean_yes_no_keywords(series):
    """Lowercase, strip, then: contains 'yes' → Yes, contains 'no' → No, else Unknown."""
    cleaned = series.str.lower().str.strip()
    return cleaned.apply(
        lambda x: 'Yes' if 'yes' in str(x)
        else ('No' if 'no' in str(x) else 'Unknown')
    ).fillna('Unknown')

keyword_yes_no_cols = [
    'Renewal_Impact_Due_to_Price_Increase',
    'Explicit_Competitor_Mention',
    'Explicit_Switching_Intent',
    'Mentioned_Competitors',
    'Price_Switching_Mentioned',
    'Discount_Offered',
]

for col in keyword_yes_no_cols:
    renewal_calls_df[col] = clean_yes_no_keywords(renewal_calls_df[col])

# Discount_or_Waiver_Requested: pre-replace '[Yes/No] ' → 'Unknown', then same keyword logic
renewal_calls_df['Discount_or_Waiver_Requested'] = renewal_calls_df['Discount_or_Waiver_Requested'].replace({
    "[Yes/No] " : "Unknown"
})
renewal_calls_df['Discount_or_Waiver_Requested'] = clean_yes_no_keywords(renewal_calls_df['Discount_or_Waiver_Requested'])

all_keyword_cols = keyword_yes_no_cols + ['Discount_or_Waiver_Requested']
print(f"Cleaned {len(all_keyword_cols)} columns (keyword yes/no match) ✓")
for col in all_keyword_cols:
    print(f"  {col}: {renewal_calls_df[col].value_counts().to_dict()}")

In [0]:
# ── Competitor_Value_Comparison: Custom keyword mapping ──
renewal_calls_df["Competitor_Value_Comparison"] = (
    renewal_calls_df["Competitor_Value_Comparison"]
    .astype(str)
    .str.lower()
    .str.strip()
)

def map_to_binary(x):
    if any(k in x for k in ['better value', 'similar value']):
        return 'Yes'
    elif any(k in x for k in [
        'not applicable',
        'not discussed',
        'not mentioned',
        'not disclosed'
    ]) or (' no ' in f" {x} "):
        return 'No'
    else:
        return 'Unknown'

renewal_calls_df['Competitor_Value_Comparison'] = renewal_calls_df['Competitor_Value_Comparison'].apply(map_to_binary)

print(f"Competitor_Value_Comparison: {renewal_calls_df['Competitor_Value_Comparison'].value_counts().to_dict()}")

In [0]:
# ── Competitor_Benefits_Mentioned: Custom keyword mapping ──
renewal_calls_df['Competitor_Benefits_Mentioned'] = (
    renewal_calls_df['Competitor_Benefits_Mentioned']
    .astype(str)
    .str.lower()
    .str.strip()
)

def map_benefits(x):
    if any(k in x for k in [
        'discount',
        'better service',
        'better value',
        'cheaper',
        'offering',
        'more work',
        'benefit'
    ]):
        return 'Yes'
    elif any(k in x for k in [
        'not applicable',
        'not discussed',
        'not mentioned',
        'no benefit'
    ]):
        return 'No'
    else:
        return 'Unknown'

renewal_calls_df['Competitor_Benefits_Mentioned'] = renewal_calls_df['Competitor_Benefits_Mentioned'].apply(map_benefits)

print(f"Competitor_Benefits_Mentioned: {renewal_calls_df['Competitor_Benefits_Mentioned'].value_counts().to_dict()}")

In [0]:
# ── Group 5: Fill NaN → 'Unknown' (already-clean columns) ──
fillna_cols = [
    'Topic_Introduced_By',
    'Percentage_Price_Increase_Mentioned',
    'Monetary_Price_Increase_Mentioned',
    'Customer_Asked_For_Justification',
    'Customer_Response',
    'Price_Range_Mentioned',
]

for col in fillna_cols:
    if renewal_calls_df[col].dtype.name == 'category':
        if 'Unknown' not in renewal_calls_df[col].cat.categories:
            renewal_calls_df[col] = renewal_calls_df[col].cat.add_categories('Unknown')
    renewal_calls_df[col] = renewal_calls_df[col].fillna('Unknown')

print(f"Filled NaN → 'Unknown' for {len(fillna_cols)} columns ✓")
for col in fillna_cols:
    print(f"  {col}: nulls remaining = {renewal_calls_df[col].isnull().sum()}")

In [0]:
# ── Desire_To_Cancel: Keyword mapping → Renewed / Desired to Cancel / Not Discussed / Unknown ──
renewal_calls_df['Desire_To_Cancel'] = (
    renewal_calls_df['Desire_To_Cancel']
    .astype(str)
    .str.lower()
    .str.strip()
)

def map_desire_to_cancel(x):
    if any(k in x for k in ['not discussed', 'not applicable']):
        return 'Not Discussed'
    elif 'cancel' in x:
        return 'Desired to Cancel'
    elif 'renew' in x:
        return 'Renewed'
    else:
        return 'Unknown'

renewal_calls_df['Desire_To_Cancel'] = renewal_calls_df['Desire_To_Cancel'].apply(map_desire_to_cancel)

print(f"Desire_To_Cancel: {renewal_calls_df['Desire_To_Cancel'].value_counts().to_dict()}")

In [0]:
# ── Convert low-cardinality object columns to 'category' dtype ──
CATEGORICAL_THRESHOLD = 30  # columns with <= 30 unique values

categorical_cols = []
for col in renewal_calls_df.select_dtypes(include='object').columns:
    if renewal_calls_df[col].nunique() <= CATEGORICAL_THRESHOLD:
        categorical_cols.append(col)

renewal_calls_df[categorical_cols] = renewal_calls_df[categorical_cols].astype('category')

print(f'Converted {len(categorical_cols)} columns to category:')
for c in categorical_cols:
    cats = renewal_calls_df[c].cat.categories.tolist()
    print(f'  {c:55s} → {len(cats)} categories: {cats[:8]}{", ..." if len(cats) > 8 else ""}')

In [0]:
# ── Final dtype summary ──
renewal_calls_df.info()

In [0]:
renewal_calls_df.to_csv("../../data/processed/renewal_calls.csv", index=False)